In [1]:
#set things
#experiment_name = "test"
#config??
#modell?



#next:
#gender labeling needs to be fixed (he! = neutral)
#get eval working
#get saving eval working

In [2]:
# ============================================================
# Baseline experiment
#
# BERT + LoRA
# No bias mitigation
# ============================================================


# ============================================================
# SETUP
# ============================================================

# Only needed in Google Colab:



from pathlib import Path
import subprocess

repo = Path("/content/gender-bias-hate-speech")

if not repo.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/load-luna/gender-bias-hate-speech.git",
         str(repo)],
        check=True,
    )

%cd /content/gender-bias-hate-speech
%pip install -r requirements.txt
#only because of colab error
%pip uninstall -y torchao


c:\content\gender-bias-hate-speech
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
#for google colab
#clone repo and install requirements


from pathlib import Path
import subprocess
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo = Path("/content/gender-bias-hate-speech")
    if not repo.exists():
        subprocess.run(
            ["git", "clone",
             "https://github.com/load-luna/gender-bias-hate-speech.git",
             str(repo)],
            check=True,
        )
else:
    # Settig root localy.
    repo = Path(r"C:\Users\lunaj\Desktop\gender-bias-hate-speech")


os.chdir(repo)
results_dir = repo / "results"
results_dir.mkdir(exist_ok=True)

print(f"Project root: {Path.cwd()}")
print("Saving results to:", results_dir)

Project root: C:\Users\lunaj\Desktop\gender-bias-hate-speech
Saving results to: C:\Users\lunaj\Desktop\gender-bias-hate-speech\results


In [4]:
# 2. IMPORTS
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from transformers import set_seed
from peft import PeftConfig

from src.config import ExperimentConfig, prepare_run

from src.data.dataset import load_data_from_config, prepare_dataset, split_dataset
from src.data.text_processing import deep_clean_tweet
from src.data.gender_labeling import add_gender_label, count_label_distribution

from src.data.splits import load_or_create_splits

from src.training.model import (
    load_model, load_tokenizer,
    tokenize_dataset,
    load_fine_tuned_model,
    prepare_lora,
    compute_metrics,
    predict_dataset,
)
from src.training.train import (
    create_training_arguments,
    create_trainer,
)

from src.evaluation.metrics import classification_metrics
from src.evaluation.fairness import equalized_odds_gaps

In [5]:
# 3. LOAD CONFIGURATION
# ============================================================

config = ExperimentConfig.from_files(
    model="configs/model_baseline_davidson_binary_v1.yaml",
    training="configs/training.yaml",
    evaluation="configs/evaluation.yaml",
    dataset="configs/datasets/davidson_binary.yaml",
    mitigations=[
        "configs/mitigation/baseline.yaml",
    ],
)

#set variable about whether the model is fine-tuned or not
is_fine_tuned = config["model"].get("is_fine_tuned", False)

print("Resolved configuration:")
print(config.to_dict())


Resolved configuration:
{'model': {'name': 'baseline_davidson_binary_v1', 'is_fine_tuned': True, 'path': 'models/baseline_davidson_binary_v1'}, 'lora': {'enabled': True, 'r': 8, 'alpha': 16, 'dropout': 0.1, 'target_modules': ['query', 'value'], 'bias': 'none', 'task_type': 'SEQ_CLS', 'modules_to_save': ['classifier']}, 'training': {'learning_rate': 2e-05, 'batch_size': 16, 'epochs': 3, 'weight_decay': 0.01, 'warmup_steps': 0.0, 'seed': 42, 'metric_for_best_model': 'f1_macro', 'greater_is_better': True, 'load_best_model_at_end': True}, 'evaluation': {'batch_size': 32, 'metrics': ['accuracy', 'f1', 'tpr_gap', 'equalized_odds'], 'counterfactual_testing': True}, 'dataset': {'name': 'davidson', 'config_name': None, 'text_column': 'text', 'label_column': 'label', 'gender_column': 'gender_label', 'source': 'local', 'split': {'directory': 'data/processed/splits/davidson_binary_v1', 'seed': 42, 'preprocessing_version': 'deep_clean_tweet_v1', 'train_split_size': 0.8, 'validation_split_size': 0.1

In [ ]:
# 4. CREATE RUN
# ============================================================



if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    results_root = Path(
        "/content/drive/MyDrive/gender-bias-hate-speech/results"
    )
else:
    results_root = repo / "results"

config, run_dir = prepare_run(
    experiment_name="baseline_davidson_binary_v1",
    config=config,
    results_root=results_root,
)

print(f"Run directory: {run_dir}")


Run directory: C:\Users\lunaj\Desktop\gender-bias-hate-speech\results\baseline_dataset_a\run_014


In [7]:
# 5. LOAD DATASET
# ============================================================

dataset = load_data_from_config(config)
dataset = prepare_dataset(dataset, config)

print(dataset)



Map:   0%|          | 0/24783 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'category'],
    num_rows: 24783
})


In [8]:
# 6. Clean dataset // text preprocessing
# ============================================================

dataset = dataset.map(deep_clean_tweet)


# Remove tweets reduced to empty text (e.g. URL-only tweets).
rows_before_filter = len(dataset)
dataset = dataset.filter(lambda example: bool(example["text"].strip()))
print(f"Removed {rows_before_filter - len(dataset)} empty texts after cleaning.")


Map:   0%|          | 0/24783 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24783 [00:00<?, ? examples/s]

Removed 0 empty texts after cleaning.


In [9]:
# 7. add Gender Label
# ============================================================
dataset = dataset.map(add_gender_label)

print("Gender label distribution:")
count_label_distribution(dataset, config)

### add more analysis....


Map:   0%|          | 0/24783 [00:00<?, ? examples/s]

Gender label distribution:
           ['neither', 'harmful']
male       [524, 2010]
female     [235, 2585]
equal      [32, 264]
none       [3372, 15761]


[[524, 2010], [235, 2585], [32, 264], [3372, 15761]]

In [10]:

# Split dataset
dataset = load_or_create_splits(
    dataset, config, run_dir, copy_assignments=True,
)

In [11]:
# 7. Preprocessing MITIGATION
# ============================================================

mitigation_config = config.get(
    "mitigation",
    {}
)

print("\nMitigation configuration:")
print(mitigation_config)




Mitigation configuration:
{}


In [12]:
# 6. DATASET SANITY CHECKS
# ============================================================

print("\nDataset splits:")

for split in dataset.keys():
    print(f"{split}: {len(dataset[split])}")


print("\nColumns:")
print(dataset["train"].column_names)


print("\nFirst training example:")
print(dataset["train"][0])


# Target label distribution
if "label" in dataset["train"].column_names:

    print("\nLabel distribution:")

    print(
        dataset["train"]
        .to_pandas()["label"]
        .value_counts()
        .sort_index()
    )


# Gender-label distribution
if "gender_label" in dataset["train"].column_names:

    print("\nGender distribution:")

    print(
        dataset["train"]
        .to_pandas()["gender_label"]
        .value_counts()
        .sort_index()
    )




Dataset splits:
train: 19815
validation: 2492
test: 2476

Columns:
['text', 'label', 'category', 'gender_label']

First training example:
{'text': "&#128514;&#128514;fucc the extra shit i'm seeing that hoe this weekend", 'label': 1, 'category': 'harmful', 'gender_label': 3}

Label distribution:
label
0     3330
1    16485
Name: count, dtype: int64

Gender distribution:
gender_label
0     2016
1     2246
2      244
3    15309
Name: count, dtype: int64


In [13]:
#check fine-tuned model


if is_fine_tuned:
    model_path=config["model"]["path"]
    model_config = PeftConfig.from_pretrained(model_path)
    model_folder = Path(model_path)
    if not (model_folder / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"Keine adapter_config.json in {model_folder}. "
            "Bitte das gespeicherte Modell als Input hinzufügen und model_path anpassen."
        )
    if not any((model_folder / name).is_file() for name in
               ("adapter_model.safetensors", "adapter_model.bin")):
        raise FileNotFoundError(f"Keine LoRA-Gewichte in {model_folder} gefunden.")

    print(f"Model path: {model_path}")
    print(f"Model config: {model_config}")

Model path: models/baseline_davidson_binary_v1
Model config: LoraConfig(task_type='SEQ_CLS', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.1', base_model_name_or_path='google-bert/bert-base-uncased', revision=None, inference_mode=True, r=8, target_modules={'query', 'value'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=['classifier', 'classifier', 'score'], init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=No

In [14]:
# 8. LOAD TOKENIZER
# ============================================================

model_path = config["model"]["path"]

tokenizer_path = model_path

if is_fine_tuned and not (Path(model_path) / "tokenizer_config.json").exists():
    saved_config = PeftConfig.from_pretrained(model_path)
    tokenizer_path = saved_config.base_model_name_or_path
    if not tokenizer_path:
        raise ValueError("In adapter_config.json fehlt base_model_name_or_path.")

tokenizer = load_tokenizer(tokenizer_path)

print("\nTokenizer loaded:")
print(tokenizer.__class__.__name__)






Tokenizer loaded:
BertTokenizer


In [15]:
# 9. TOKENIZE DATASET
# ============================================================

tokenized_dataset = tokenize_dataset(
    dataset=dataset,
    tokenizer=tokenizer,

)

print("\nTokenized dataset:")
print(tokenized_dataset)

Map:   0%|          | 0/19815 [00:00<?, ? examples/s]

Map:   0%|          | 0/2492 [00:00<?, ? examples/s]

Map:   0%|          | 0/2476 [00:00<?, ? examples/s]


Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 19815
    })
    validation: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2492
    })
    test: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2476
    })
})


In [16]:
# 10. LOAD MODEL
# ============================================================

# load_model() should:
#
# 1. load bert-base-uncased
# 2. create the sequence-classification head
# 3. apply LoRA
#
# LoRA is part of the standard model setup and therefore
# should happen inside src/training/model.py.


set_seed(config["training"]["seed"])

label_names = {
    int(key): value
    for key, value in config["dataset"]["label_mapping"]["names"].items()
}

#if fine-tuned model exists, load it. Otherwise, load the base model and prepare it for LoRA fine-tuning.
is_fine_tuned = config["model"].get("is_fine_tuned", False)

if is_fine_tuned:
    lora_model = load_fine_tuned_model(
        model_path=config["model"]["path"],
        label_names=label_names
    )
else:


    model = load_model(
        model_path=config["model"]["path"],
        label_names=label_names,
    )

    lora_model = prepare_lora(
        model=model,
        config=config
    )

print("\nModel loaded:")
print(lora_model.__class__.__name__)


# Check that LoRA is actually active.
if hasattr(lora_model, "print_trainable_parameters"):

    print("\nTrainable parameters:")

    lora_model.print_trainable_parameters()




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded:
PeftModelForSequenceClassification

Trainable parameters:
trainable params: 0 || all params: 109,780,228 || trainable%: 0.0000


In [17]:
# 11. TRAINING ARGUMENTS
# ============================================================

training_args = create_training_arguments(
    config=config,
    output_dir = run_dir / "checkpoints",
)

print("\nTraining arguments:")
print(training_args)




Training arguments:
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalS

In [18]:
# 12. CREATE TRAINER
# ============================================================


trainer = create_trainer(
    model=lora_model,
    tokenizer=tokenizer,
    training_args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics= compute_metrics
)

print("\nTrainer created.")




Trainer created.


In [19]:
# 13. TRAIN MODEL
# ============================================================

if not is_fine_tuned:
    train_result = trainer.train()
    print("\nTraining finished.")

    print("\nTraining metrics:")
    print(train_result.metrics)







In [20]:
## save Modell


if not is_fine_tuned:
    print("Speichere Modell …")
    model_dir = run_dir / "model"
    trainer.save_model(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
else:
    print("ist schon gespeichert")

ist schon gespeichert


In [21]:
## predictions on test set

test_dataset = predict_dataset(tokenized_dataset["test"],trainer)
#show label distribution on test set
count_label_distribution(dataset["test"], config)

c:\Users\lunaj\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# Grouped evaluation: any number of single-label classes
from src.evaluation.report import evaluate_groups, save_evaluation_report
from IPython.display import display, Image

metrics = evaluate_groups(
    test_dataset,
    label_names=lora_model.config.id2label,
    comparison=("female", "male"),
)

# Save immediately so the report does not depend on subsequent cells.
metrics_path = save_evaluation_report(metrics, run_dir)
summary_df = pd.DataFrame([
    {"group": group, **values["summary"]}
    for group, values in metrics["groups"].items()
])
class_metrics_df = pd.DataFrame([
    {"group": group, **row}
    for group, values in metrics["groups"].items()
    for row in values["per_class"]
])
display(summary_df)
display(class_metrics_df)
display(Image(filename=str(run_dir / "confusion_matrices.png")))
print(f"Evaluation saved to: {run_dir}")


In [ ]:
# Per-class one-vs-rest fairness: female minus male
# Undefined rates/gaps remain null (shown as NaN in a DataFrame).
# These are descriptive subgroup differences, not proof of causal bias.
fairness_df = pd.DataFrame(metrics["fairness"]["per_class"])
display(fairness_df)


In [ ]:
# 17. SAVE INDIVIDUAL PREDICTIONS
# ============================================================

prediction_data = {
    "text": dataset["test"]["text"],
    "gender_label": dataset["test"]["gender_label"] if "gender_label" in dataset["test"].column_names else None,
    "label": test_dataset["all"]['labels'],
    "prediction": test_dataset["all"]['predictions'],
}



predictions_df = pd.DataFrame(
    prediction_data
)


predictions_path = (
    run_dir / "predictions.csv"
)


predictions_df.to_csv(
    predictions_path,
    index=False,
)


print(
    f"\nPredictions saved to: "
    f"{predictions_path}"
)


In [ ]:
# All group/class metrics, fairness gaps and matrices were saved together.
print(f"Complete report: {metrics_path}")
print("Tables: group_metrics.csv, class_metrics.csv, fairness_metrics.csv")


In [ ]:
print("EVALUATION FINISHED")
print(f"Run directory: {run_dir}")
display(summary_df)
print(f"Config: {run_dir / 'config.yaml'}")
print(f"Metrics: {metrics_path}")
print(f"Predictions: {predictions_path}")
print(f"Confusion matrices: {run_dir / 'confusion_matrices.png'}")


In [ ]:
## comit results
"""""
!git status
!git add results/
!git commit -m "Add experiment results"
!git push
"""""

## Counterfactual Testing: Gender-Swap des Testdatensatzes

Wir verwenden denselben Testsplit und dasselbe trainierte Modell. Jeder Originaltext
behält einen zugeordneten geswappten Text und sein ursprüngliches Klassenlabel.
Gender-Gruppen werden für die geswappten Texte neu bestimmt. Unveränderte Texte
bleiben im vollständigen Testset, werden aber aus der paarweisen Swap-Auswertung ausgeschlossen.

Die lexikalischen Swaps setzen voraus, dass das Klassenlabel gültig bleibt.
Sie sind nicht immer grammatisch oder semantisch gleichwertig (z. B. `her` → `him`).
Die gespeicherten Textpaare sollten deshalb stichprobenartig geprüft werden.
Die Ergebnisse messen die Sensitivität gegenüber diesen Ersetzungen.

Hinweis: Gender-Swap und Gender-Labeling verwenden dieselbe Worterkennung.
URLs (auch www.) und @Nutzernamen werden entfernt. Beim Swap bleiben die
übrigen Satzzeichen, inneren Abstände und die übliche Groß-/Kleinschreibung erhalten.
Der Vergleich erfasst zusätzlich zum Gender-Swap auch entfernte URLs und Nutzernamen
sowie entfernte äußere Leerzeichen. Diese Änderungen zählen ebenfalls als
veränderte Texte (`swap_changed`).


In [ ]:
# 18. CREATE AND SAVE THE SWAPPED TEST DATASET
from src.evaluation.counterfactual import (
    create_swapped_test_dataset, evaluate_counterfactual_pairs,
)

counterfactual_dir = run_dir / "counterfactual"
counterfactual_dir.mkdir(parents=True, exist_ok=True)
original_test = dataset["test"]
swapped_test = create_swapped_test_dataset(original_test)
# Save before tokenization, including source metadata and the paired original text.
swapped_test.to_csv(str(counterfactual_dir / "swapped_test.csv"), index=False)
print(f"Test rows: {len(swapped_test)}; changed texts: {sum(swapped_test['swap_changed'])}")
display(swapped_test.to_pandas().loc[
    lambda frame: frame["swap_changed"],
    ["pair_id", "original_text", "text", "label", "original_gender_label", "gender_label"],
].head(10))


In [ ]:
# 19. RETOKENIZE AND EVALUATE THE ENTIRE SWAPPED TEST SET
# Retokenize from swapped text; never reuse the original input_ids.
swapped_tokenized = tokenize_dataset(swapped_test, tokenizer)
swapped_predictions = predict_dataset(swapped_tokenized, trainer)
cf_label_names = trainer.model.config.id2label
swapped_metrics = evaluate_groups(swapped_predictions, cf_label_names)
save_evaluation_report(swapped_metrics, counterfactual_dir / "swapped_full")

# Full-set groups here describe the gender terms AFTER swapping.
display(pd.DataFrame([
    {"group_after_swap": group, **values["summary"]}
    for group, values in swapped_metrics["groups"].items()
]))
display(pd.DataFrame([
    {"group_after_swap": group, **row}
    for group, values in swapped_metrics["groups"].items()
    for row in values["per_class"]
]))
display(pd.DataFrame(swapped_metrics["fairness"]["per_class"]))
display(Image(filename=str(counterfactual_dir / "swapped_full" / "confusion_matrices.png")))


In [ ]:
# 20. PAIRED COMPARISON ON THE SAME EXAMPLES
# Existing original predictions are in test-split order; verify labels and length.
np.testing.assert_array_equal(test_dataset["all"]["labels"], original_test["label"])
np.testing.assert_array_equal(swapped_predictions["all"]["labels"], original_test["label"])
paired_metrics, changed_reports, paired_predictions = evaluate_counterfactual_pairs(
    swapped_test,
    test_dataset["all"]["predictions"],
    swapped_predictions["all"]["predictions"],
    cf_label_names,
)
paired_predictions.to_csv(counterfactual_dir / "paired_predictions.csv", index=False)
(counterfactual_dir / "paired_metrics.json").write_text(
    json.dumps(paired_metrics, indent=2, ensure_ascii=False, allow_nan=False),
    encoding="utf-8",
)
paired_summary = pd.DataFrame(paired_metrics["groups"]).drop(columns="prediction_transitions")
paired_summary.to_csv(counterfactual_dir / "paired_summary.csv", index=False)
display(paired_summary)

# Both reports use ONLY changed texts and the SAME ORIGINAL gender groups.
# Thus male means originally male (usually swapped toward female), and vice versa.
comparison_rows = []
for variant, report in changed_reports.items():
    save_evaluation_report(report, counterfactual_dir / variant)
    for group, values in report["groups"].items():
        comparison_rows.append({"variant": variant, "original_group": group, **values["summary"]})
    print(variant)
    display(pd.DataFrame([
        {"original_group": group, **row}
        for group, values in report["groups"].items()
        for row in values["per_class"]
    ]))
    display(pd.DataFrame(report["fairness"]["per_class"]))
    display(Image(filename=str(counterfactual_dir / variant / "confusion_matrices.png")))
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(counterfactual_dir / "changed_comparison.csv", index=False)
display(comparison_df)
print(f"Counterfactual evaluation saved to: {counterfactual_dir}")
